## Carbon Intensity Pipeline Functions
This section defines reusable functions for loading and preprocessing Carbon Intensity API data.

In [1]:
import requests
import pandas as pd
from datetime import timedelta

In [2]:
def load_carbon_intensity_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Load Carbon Intensity API data for a specified date range.
    Handles the API 31-day limit by pulling data in chunks.
    """
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    if start >= end:
        raise ValueError("end_date must be after start_date")

    dfs = []
    current = start

    while current < end:
        next_date = min(current + timedelta(days=30), end)

        url = (
            f"https://api.carbonintensity.org.uk/intensity/"
            f"{current.strftime('%Y-%m-%d')}/{next_date.strftime('%Y-%m-%d')}"
        )

        response = requests.get(url, timeout=30)
        response.raise_for_status()

        data = response.json().get("data", [])
        if data:
            dfs.append(pd.json_normalize(data))

        current = next_date

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)

In [3]:
def preprocess_carbon_intensity_data(
    df: pd.DataFrame,
    filter_start: str | None = None,
    filter_end: str | None = None
) -> pd.DataFrame:
    """
    Clean and preprocess Carbon Intensity API data.

    Output columns:
    - datetime
    - carbon_intensity/gCO2/kWh
    """
    if df.empty:
        return pd.DataFrame(columns=["datetime", "carbon_intensity/gCO2/kWh"])

    df = df.copy()

    required_cols = {"from", "intensity.actual", "intensity.forecast"}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    df = df.rename(columns={
        "intensity.actual": "actual",
        "intensity.forecast": "forecast"
    })

    df["datetime"] = pd.to_datetime(df["from"], utc=True, errors="coerce")
    df["datetime"] = df["datetime"].dt.tz_localize(None)

    if filter_start is not None:
        df = df[df["datetime"] >= pd.to_datetime(filter_start)]

    if filter_end is not None:
        df = df[df["datetime"] < pd.to_datetime(filter_end)]

    df = df.drop(columns=["forecast"], errors="ignore")

    df = df.rename(columns={"actual": "carbon_intensity/gCO2/kWh"})

    df = df[["datetime", "carbon_intensity/gCO2/kWh"]]

    df = (
        df
        .sort_values("datetime")
        .drop_duplicates(subset="datetime")
        .reset_index(drop=True)
    )

    return df

In [4]:
df_raw = load_carbon_intensity_data("2025-01-01", "2026-01-01")
df_raw.head()

,from,to,intensity.forecast,intensity.actual,intensity.index
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low


In [5]:
df_clean = preprocess_carbon_intensity_data(
    df_raw,
    filter_start="2025-01-01",
    filter_end="2026-01-01"
)

df_clean.head()

,datetime,carbon_intensity/gCO2/kWh
0,2025-01-01 00:00:00,55
1,2025-01-01 00:30:00,54
2,2025-01-01 01:00:00,53
3,2025-01-01 01:30:00,53
4,2025-01-01 02:00:00,47


In [6]:
df_clean.shape

(17520, 2)

In [7]:
df_clean.isnull().sum()

datetime                     0
carbon_intensity/gCO2/kWh    0
dtype: int64

In [8]:
df_clean.duplicated().sum()

np.int64(0)

In [9]:
df_clean["datetime"].diff().value_counts().head()

datetime
0 days 00:30:00    17519
Name: count, dtype: int64

In [10]:
# df_clean["datetime"].diff().value_counts().head()

In [11]:
import carbon_pipeline as cp
df = cp.load_carbon_intensity_data("2025-01-01", "2026-01-01")

ModuleNotFoundError: No module named 'carbon_pipeline'

In [ ]:
df.head()

,from,to,intensity.forecast,intensity.actual,intensity.index
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low


In [ ]:
df_raw = load_carbon_intensity_data("2025-01-01", "2026-01-01")

df_clean = preprocess_carbon_intensity_data(df_raw)

df_clean.head()

,datetime,carbon_intensity/gCO2/kWh
0,2024-12-31 23:30:00,51
1,2025-01-01 00:00:00,55
2,2025-01-01 00:30:00,54
3,2025-01-01 01:00:00,53
4,2025-01-01 01:30:00,53


In [ ]:
# df_clean.to_csv("carbon_intensity_2025_lean.csv", index=False)